In [77]:
class Market():
    def __init__(self: "Market", universe: list[str], latest_timestamp: str) -> None:
        self.universe: list[str] = universe
        self.quotes: dict[str, dict] = {}  # {key: product, value: {key: timestep, value: price}}
        self.now = latest_timestamp
        
    def update(self: "Market", quote: dict)-> None:
        if quote['id'] != "Clock":
            self.quotes[quote['id']] = quote


    def get_current_price(self, asset): 
        if asset not in self.quotes:
            raise ValueError(f"No quote available for {asset}")
        
        return self.quotes[asset][self.now]

    

    def __str__(self: "Market") -> str:
        return str(self.quotes)

In [ ]:

"""
Portfolio management module for trading simulation.

This module defines the Portfolio class, which manages cash and positions,
enforcing leverage limits and providing methods to buy and sell products.

Author: Mathis Makarski
Date: 2025-11-02
"""

import logging

#logger = logging.getLogger("local_eval")

class Portfolio():
    def __init__(self, cash: float, market: "Market", leverage_limit: float) -> None:
        self.cash: float = cash
        self.market: Market = market
        self.positions: dict[str, int] = {}  # key: product, value: quantity
        self.leverage_limit: float = leverage_limit  # max leverage allowed

    def _get_price(self,  product: str) -> float:
        """Retrieve the last market price for a given product."""
        if product not in self.market.quotes:
            raise ValueError(f"No quote available for {product}")
        #return self.market.quotes[product].get("price", None)
        return self.market.quotes[product].get("price", None)
    
    def _get_timestamp(self, product) -> int:
        """Retrieve the current market timestamp for the product quote."""
        if product not in self.market.quotes:
            raise ValueError(f"No quote available for {product}")
        return self.market.quotes[product].get("timestep", None)

    def get_gross_exposure(self): 
        total = 0.0
        for product, qty in self.positions.items(): 
            price = self.market.get_current_price(product)
            total += abs(qty) * price
        return total

    def _gross_exposure(self) -> float:
        """Compute gross exposure = sum(|position| * price)"""
        total = 0.0
        for product, qty in self.positions.items():
            price = self._get_price(product)
            total += abs(qty) * price
        return total

    def get_net_asset_value(self): 
        """Compute portfolio net asset value = cash + sum(qty * price)"""
        value = self.cash
        for product, qty in self.positions.items():
            price = self.market.get_current_price(product)
            value += qty* price

        return value

    def _net_asset_value(self) -> float:
        """Compute portfolio net asset value = cash + sum(qty * price)"""
        value = self.cash
        for product, qty in self.positions.items():
            price = self._get_price(product)
            value += qty * price
        return value
    
    def _leverage(self) -> float:
        """Compute current leverage = gross exposure / net asset value"""
        gross = self._gross_exposure()
        net_value = self._net_asset_value()
        return gross / max(net_value, 1e-8)  # Avoid division by zero

    def _check_leverage(self, new_cash: float, new_positions: dict[str, int]) -> bool:
        """Check whether the new portfolio state respects leverage limits."""
        gross = sum(abs(qty) * self._get_price(p) for p, qty in new_positions.items())
        net_value = new_cash + sum(qty * self._get_price(p) for p, qty in new_positions.items())
        leverage = gross / max(net_value, 1e-8)
        return leverage <= self.leverage_limit

    def buy(self, product: str, quantity: int) -> bool:
        """Attempt to buy `quantity` units of `product`."""
        timestamp = self._get_timestamp(product)
        price = self._get_price(product)
        cost = price * quantity

        new_cash = self.cash - cost
        new_positions = self.positions.copy()
        new_positions[product] = new_positions.get(product, 0) + quantity

        if not self._check_leverage(new_cash, new_positions):
            #logger.warning(f"{timestamp} | Trade rejected: leverage limit exceeded.")
            return False

        self.cash = new_cash
        self.positions = new_positions
        #logger.info(f"{timestamp} | BOUGHT {quantity} {product} @ {price} | new cash={self.cash:.2f}")
        return True

    def sell(self, product: str, quantity: int) -> bool:
        """Attempt to sell `quantity` units of `product` (shorts allowed)."""
        timestamp = self._get_timestamp(product)
        price = self._get_price(product)
        proceeds = price * quantity

        new_cash = self.cash + proceeds
        new_positions = self.positions.copy()
        new_positions[product] = new_positions.get(product, 0) - quantity

        if not self._check_leverage(new_cash, new_positions):
            #logger.warning("Trade rejected: leverage limit exceeded.")
            return False

        self.cash = new_cash
        self.positions = new_positions
        #logger.info(f"{timestamp} | SOLD {quantity} {product} @ {price} | new cash={self.cash:.2f}")
        return True

    def summary(self) -> dict:
        """Return a snapshot of the portfolio."""
        return {
            "cash": self.cash,
            "positions": self.positions,
            "gross_exposure": self._gross_exposure(),
            "net_value": self._net_asset_value(),
            "leverage": self._leverage(),
        }
    

    def compute_daily_navs(self, dates): 
        tmp_date = self.market.now
        navs = {} #: format date:nav
        for date in dates: 
            self.market.now = date
            navs[date] = self.get_net_asset_value()

        self.market.now = tmp_date #: reset original date
        return navs


        

    def __str__(self) -> str:
        return str(self.summary())

In [79]:
import pandas as pd

data = pd.read_csv("../data/sp500_close.csv")



In [80]:

tickers = data["Ticker"].unique()
market = Market(universe=list(tickers), latest_timestamp="")

for ticker, ticker_df in data.groupby("Ticker"):
    d = (ticker_df
         .set_index("Date")["Price Close"]
         .to_dict())

    market.quotes[ticker] = d
    market.now = ticker_df["Date"].iloc[-1]


In [ ]:


universe = list(market.universe)
asset = universe[1] 
dates = list(market.quotes[asset].keys())



pf= Portfolio(
    cash = 10000, 
    market = market, 
    leverage_limit =1 
)
initial_position = {asset:10}
pf.positions = initial_position




for date in dates[:10]: 
    pf.market.now = date
    print(f"grosse exposure at {date}: {pf.get_gross_exposure()}")
    print(f"net asset value at {date}: {pf.get_net_asset_value()}")


grosse exposure at 2000-01-03: 450.0
net asset value at 2000-01-03: 10450.0
grosse exposure at 2000-01-04: 431.25
net asset value at 2000-01-04: 10431.25
grosse exposure at 2000-01-05: 430.0
net asset value at 2000-01-05: 10430.0
grosse exposure at 2000-01-06: 442.5
net asset value at 2000-01-06: 10442.5
grosse exposure at 2000-01-07: 441.875
net asset value at 2000-01-07: 10441.875
grosse exposure at 2000-01-10: 427.5
net asset value at 2000-01-10: 10427.5
grosse exposure at 2000-01-11: 416.25
net asset value at 2000-01-11: 10416.25
grosse exposure at 2000-01-12: 423.75
net asset value at 2000-01-12: 10423.75
grosse exposure at 2000-01-13: 428.125
net asset value at 2000-01-13: 10428.125
grosse exposure at 2000-01-14: 440.625
net asset value at 2000-01-14: 10440.625
